# Signal study: does the Consumer-Trend Score predict anything?

Section 11.4 of the build spec. **Point-in-time discipline:** each score uses only data dated on or before the score date; forward returns use closes strictly after it. Whatever the result, it is reported honestly; a null or insufficient-data result is the expected outcome on a synthetic corpus.

_Notebook generated by `scripts/build_signal_notebook.py` on 2026-10-27; re-run that script to refresh the outputs._

In [1]:
import sys
sys.path.insert(0, '../src')
from pathlib import Path
exec(Path('../evals/signal_study.py').read_text())  # defines run_study()
from IPython.display import Markdown
Markdown(run_study())

## Signal study: point-in-time Consumer-Trend Score vs forward returns

_Run 2026-10-27 on the fixture corpus (synthetic social data, real closes for 10 tickers recorded 2026-10-27)._

- Score dates: 2026-09-15, 2026-09-22, 2026-09-29 (weekly, point-in-time).
- Investable trend signals produced: 63; with a price fixture: 27 (8 distinct tickers).

| Horizon | Top-tercile mean excess return vs SPY (95% CI) | Bottom-tercile | n top / n bottom |
|---|---|---|---|
| 30d | +9.06% [+3.91%, +14.21%] | +1.67% [-8.36%, +11.70%] | 7 / 4 |
| 60d | n/a | n/a | 0 / 0 |
| 90d | n/a | n/a | 0 / 0 |

Per-observation detail:

| as of | entity | ticker | score | 30d | 60d | 90d |
|---|---|---|---|---|---|---|
| 2026-09-22 | Xbox Game Pass | MSFT | 23 | +4.09% | n/a | n/a |
| 2026-09-29 | Starbucks | SBUX | 23 | n/a | n/a | n/a |
| 2026-09-22 | Chipotle | CMG | 24 | -13.97% | n/a | n/a |
| 2026-09-29 | Xbox Game Pass | MSFT | 27 | n/a | n/a | n/a |
| 2026-09-22 | Starbucks | SBUX | 29 | +8.25% | n/a | n/a |
| 2026-09-29 | Chipotle | CMG | 29 | n/a | n/a | n/a |
| 2026-09-29 | Nike | NKE | 31 | n/a | n/a | n/a |
| 2026-09-22 | Nike | NKE | 31 | +6.30% | n/a | n/a |
| 2026-09-29 | AirPods | AAPL | 32 | n/a | n/a | n/a |
| 2026-09-15 | Starbucks | SBUX | 36 | -0.47% | n/a | n/a |
| 2026-09-29 | Costco | COST | 38 | n/a | n/a | n/a |
| 2026-09-15 | Xbox Game Pass | MSFT | 39 | +10.02% | n/a | n/a |
| 2026-09-22 | Kirkland Signature | COST | 39 | +12.55% | n/a | n/a |
| 2026-09-15 | AirPods | AAPL | 43 | +2.88% | n/a | n/a |
| 2026-09-29 | Kirkland Signature | COST | 44 | n/a | n/a | n/a |
| 2026-09-22 | Costco | COST | 45 | +12.55% | n/a | n/a |
| 2026-09-15 | Chipotle | CMG | 46 | -9.20% | n/a | n/a |
| 2026-09-22 | HOKA | DECK | 48 | +8.51% | n/a | n/a |
| 2026-09-29 | HOKA | DECK | 49 | n/a | n/a | n/a |
| 2026-09-15 | Nike | NKE | 49 | +3.80% | n/a | n/a |
| 2026-09-29 | e.l.f. | ELF | 49 | n/a | n/a | n/a |
| 2026-09-22 | AirPods | AAPL | 50 | +3.32% | n/a | n/a |
| 2026-09-15 | Kirkland Signature | COST | 50 | +9.43% | n/a | n/a |
| 2026-09-22 | e.l.f. | ELF | 51 | +14.10% | n/a | n/a |
| 2026-09-15 | Costco | COST | 57 | +9.43% | n/a | n/a |
| 2026-09-15 | HOKA | DECK | 61 | +11.06% | n/a | n/a |
| 2026-09-15 | e.l.f. | ELF | 73 | +24.30% | n/a | n/a |

### Caveats (read these before the numbers)
- **Synthetic social data.** The fixture corpus was generated with designed trend shapes; any relationship with real price moves is coincidence by construction.
- **Survivorship / selection.** Only tickers with a bundled price fixture are evaluated.
- **Tiny sample, multiple horizons.** Three horizons × two groups on a handful of names: treat every interval above as uninformative.
- **Forward windows truncated** by the 90-day close history; later horizons have fewer observations.
- **What would make this real:** run `jobs/refresh_trends.py` daily in live mode for 6+ months, persist each snapshot, then re-run this study on stored scores vs realised returns, with the same point-in-time rule and a pre-registered hypothesis (top-decile 60-day excess return > 0).


## How to turn this into a real test

1. Run `python -m options_scout.jobs.refresh_trends --mode live` daily; every snapshot is stored with its `as_of`.
2. After 6+ months, join stored scores to realised 30/60/90-day returns (yfinance) and SPY.
3. Pre-register the hypothesis (top-decile 60-day excess return > 0), report the CI, the sample size, and the number of hypotheses tested. Survivorship bias: keep delisted names in the sample.
